# KYRION — Figure & Table Generation

Regenerates every figure and LaTeX table for the paper from the frozen result
CSVs in `research/data/`. Run top to bottom. Figures are written to
`paper_figures/` at 300 dpi; the LaTeX tables print inline.

- **Figure 1** — per-model performance on the four informative metrics
  (Trust, Quality, Code smells, Dead code). The hallucination panel is
  omitted because it is flat zero across all models.
- **Figure 2** — the stylistic-signatures headline (the smell types that
  significantly separate models).
- **Table I** — master per-model results table with 95% bootstrap CIs.
- **Table II** — omnibus Kruskal-Wallis significance with Benjamini-Hochberg
  FDR correction.

Inputs: `research/data/results_runs.csv`, `research/data/smell_breakdown.csv`.

This notebook replaces the earlier `generate_paper_figures.ipynb` (v1,
superseded) and `generate_paper_figures_v4.ipynb` (consolidated here) as the
single source of truth for the paper's figures/tables.


In [ ]:
import csv
from pathlib import Path

import numpy as np
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "paper_figures" else Path.cwd()
DATA = ROOT / "research" / "data"
OUT = ROOT / "paper_figures"
OUT.mkdir(exist_ok=True)

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})


def short(m):
    return (m.split("__")[-1].replace("-instruct", "").replace("-70b", "")
             .replace("-120b", "").replace("-super-49b-v1", "-49b").replace("-v4-flash", "-flash"))


runs = list(csv.DictReader(open(DATA / "results_runs.csv")))
models = sorted(
    {r["model"] for r in runs},
    key=lambda m: -np.mean([float(r["trust_score"]) for r in runs if r["model"] == m]),
)
print(f"Loaded {len(runs)} runs, {len(models)} models")
for m in models:
    print(" ", short(m), sum(1 for r in runs if r["model"] == m), "runs")


## Helpers: bootstrap CI, task-level means, Kruskal-Wallis

In [ ]:
def boot_ci(vals, n=10000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(vals, float)
    b = rng.choice(a, size=(n, len(a)), replace=True).mean(1)
    return a.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


def vals_of(metric, model):
    return [float(r[metric]) for r in runs if r["model"] == model]


def task_means(metric):
    cell = {}
    for r in runs:
        cell.setdefault((r["model"], r["task"]), []).append(float(r[metric]))
    by = {}
    for (m, _), v in cell.items():
        by.setdefault(m, []).append(np.mean(v))
    return by


def kruskal(metric):
    by = task_means(metric)
    groups = [by[m] for m in models]
    allv = [v for g in groups for v in g]
    if len(set(allv)) <= 1:
        return float("nan"), float("nan")
    try:
        return stats.kruskal(*groups)
    except ValueError:
        return float("nan"), float("nan")


## Figure 1 — Per-model performance, four informative metrics (95% CIs)

Trust and Quality show convergence (overlapping CIs); Code smells and Dead code
show the divergence. The hallucination panel is omitted because all values are
zero.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
panels = [("trust_score", "Trust Score"), ("quality_score", "Quality Score"),
          ("code_smells", "Code Smells / gen"), ("dead_code", "Dead Code / gen")]
for ax, (mk, lab) in zip(axes.flat, panels):
    means, los, his = [], [], []
    for m in models:
        mn, lo, hi = boot_ci(vals_of(mk, m))
        means.append(mn); los.append(mn - lo); his.append(hi - mn)
    color = "#4C72B0" if mk in ("trust_score", "quality_score") else "#C44E52"
    ax.bar(range(len(models)), means, yerr=[los, his], capsize=4, color=color, alpha=.85)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([short(m) for m in models], rotation=30, ha="right", fontsize=8)
    H, p = kruskal(mk)
    tag = f"H={H:.2f}, p={p:.3f}" if p == p else ""
    ax.set_title(f"{lab}   ({tag})", fontsize=10)
fig.suptitle("Per-model performance (mean, 95% bootstrap CI). Blue: converged metrics; red: divergent.", fontsize=11)
fig.tight_layout()
fig.savefig(OUT / "fig1_performance.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig1_performance.png")


## Figure 2 — Stylistic signatures (headline)

In [ ]:
sb = list(csv.DictReader(open(DATA / "smell_breakdown.csv")))
sig = ["missing_module_docstring", "long_function", "long_line"]
labels = ["Missing module\ndocstring\n(p<1e-15)", "Long function\n(p<1e-9)", "Long line\n(p=0.008)"]
mcols = [c for c in sb[0].keys() if c != "smell_type"]
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(sig)); w = 0.16
colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(mcols)))
for i, mc in enumerate(mcols):
    vals = [float(next(r[mc] for r in sb if r["smell_type"] == t)) for t in sig]
    ax.bar(x + i * w, vals, w, label=mc, color=colors[i])
ax.set_xticks(x + w * (len(mcols) - 1) / 2)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Mean incidence per generation")
ax.legend(fontsize=8, ncol=2)
ax.set_title("Stylistic signatures that significantly separate models")
fig.tight_layout()
fig.savefig(OUT / "fig2_signatures.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig2_signatures.png")


## Table I (LaTeX) — Master results with 95% CI

In [ ]:
METRICS = [("trust_score", "Trust"), ("quality_score", "Quality"), ("code_smells", "Smells"),
           ("dead_code", "Dead"), ("hallucination_count", "Halluc")]
mm = {mk: {m: np.mean(vals_of(mk, m)) for m in models} for mk, _ in METRICS}
low = {"code_smells", "dead_code", "hallucination_count"}
print(r"\begin{table*}[t]\centering")
print(r"\caption{Per-model performance (mean; 95\% bootstrap CI). Best per column bold.}")
print(r"\label{tab:master}\begin{tabular}{l" + "c" * len(METRICS) + r"}\toprule")
print("Model & " + " & ".join(l for _, l in METRICS) + r" \\ \midrule")
for m in models:
    cells_ = []
    for mk, _ in METRICS:
        mn, lo, hi = boot_ci(vals_of(mk, m))
        best = (min if mk in low else max)(mm[mk].values())
        s = f"{mn:.1f}" + (f" [{lo:.1f},{hi:.1f}]" if mk in ("trust_score", "quality_score") else "")
        if abs(mm[mk][m] - best) < 1e-9:
            s = r"\textbf{" + s + "}"
        cells_.append(s)
    print(f"{short(m)} & " + " & ".join(cells_) + r" \\")
print(r"\bottomrule\end{tabular}\end{table*}")


## Table II (LaTeX) — Omnibus with BH correction

In [ ]:
ALLM = [("trust_score", "Trust"), ("quality_score", "Quality"), ("total_issues", "Total issues"),
        ("security_issues", "Security"), ("dead_code", "Dead code"), ("code_smells", "Code smells")]
res = []
for mk, lab in ALLM:
    H, p = kruskal(mk)
    res.append([lab, H, p])
ps = [p for _, _, p in res]
order = sorted(range(len(ps)), key=lambda i: ps[i])
bh = [0] * len(ps)
prev = 1.0
for rank, i in enumerate(reversed(order)):
    k = len(ps) - rank
    v = min(prev, ps[i] * len(ps) / k)
    bh[i] = v
    prev = v
print(r"\begin{table}[t]\centering")
print(r"\caption{Kruskal--Wallis omnibus with Benjamini--Hochberg FDR correction.}")
print(r"\label{tab:sig}\begin{tabular}{lcccc}\toprule")
print(r"Metric & $H$ & $p_{raw}$ & $p_{BH}$ & Sig? \\ \midrule")
for (lab, H, p), b in zip(res, bh):
    sig = "Yes" if b < 0.05 else "No"
    print(f"{lab} & {H:.2f} & {p:.4f} & {b:.4f} & {sig} " + r"\\")
print(r"Hallucination & -- & -- & -- & No variance " + r"\\")
print(r"\bottomrule\end{tabular}\end{table}")
